# Preprocessing de imágenes dermatoscópicas (pasos 1-5)

Pipeline hasta la imagen **sin blur** (para features) y **con blur** (para segmentar). No incluye la segmentación.

| Paso | Qué hace |
|---|---|
| 1 | Detecta y recorta los bordes negros de la lupa (mayor rectángulo inscrito en la zona válida) |
| 2 | Recorte cuadrado central + resize a tamaño fijo |
| 3 | Color constancy (Shades of Gray, p=6) |
| 4 | Eliminación de pelos (DullRazor: blackhat multi-orientación + filtro de forma + inpainting) |
| 5 | Dos copias: **sin blur** (features de color/textura) y **con blur** (solo para segmentar) |

Coge 2 imágenes de `train/nevus` y 2 de `train/others` y dibuja el resultado tras cada paso.

In [ ]:
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# ---------------- Configuración ----------------
DATA_DIR = None          # p. ej. Path("../train"); si es None se busca automáticamente
CLASSES = ["nevus", "others"]
N_PER_CLASS = 5
SEED = 42                # cambia el seed para ver otras imágenes
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

TARGET_SIZE = 512        # lado final (px)
DARK_THR = 35            # gris < DARK_THR => candidato a borde negro
SOG_P = 6                # norma de Minkowski de Shades of Gray
HAIR_LEN = 17            # longitud del kernel lineal (px a 512)
HAIR_THR = 10            # umbral del blackhat
BLUR_KSIZE = 5           # kernel (impar) del median blur para segmentación

In [ ]:
def find_train_dir():
    """Sube por los directorios hasta encontrar la carpeta 'train' del proyecto."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "train").is_dir() and (p / "val").is_dir():
            return p / "train"
    raise FileNotFoundError("No encuentro la carpeta train/. Pon DATA_DIR a mano.")


def read_rgb(path):
    """Lectura robusta (rutas con tildes/espacios en Windows) y en RGB."""
    data = np.fromfile(str(path), dtype=np.uint8)
    bgr = cv2.imdecode(data, cv2.IMREAD_COLOR)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def pick_images(train_dir, classes, n, seed):
    rng = random.Random(seed)
    picked = []
    for c in classes:
        files = sorted(f for f in (train_dir / c).rglob("*") if f.suffix.lower() in EXTS)
        for f in rng.sample(files, min(n, len(files))):
            picked.append((c, f))
    return picked


train_dir = Path(DATA_DIR) if DATA_DIR else find_train_dir()
samples = pick_images(train_dir, CLASSES, N_PER_CLASS, SEED)
print("train dir:", train_dir)
for c, f in samples:
    print(f"{c:7s} {f.name}")

## Funciones del pipeline

### Paso 1 · Bordes negros de la lupa

In [ ]:
def valid_mask(img, dark_thr=DARK_THR):
    """True = píxel válido. Marca como borde las zonas oscuras conectadas al marco de la imagen."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    dark = (gray < dark_thr).astype(np.uint8)
    # la apertura elimina estructuras finas oscuras (pelos) para que no cuenten como borde
    dark = cv2.morphologyEx(dark, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    _, lab = cv2.connectedComponents(dark, connectivity=4)
    border_ids = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))) - {0}
    border = np.isin(lab, list(border_ids)).astype(np.uint8)
    # pequeño margen para comerse el degradado del viñeteo
    m = max(3, int(0.01 * min(img.shape[:2])))
    border = cv2.dilate(border, np.ones((m, m), np.uint8))
    return border == 0


def largest_rectangle(mask):
    """Mayor rectángulo axis-aligned de True (algoritmo del histograma). Devuelve y0, y1, x0, x1."""
    H, W = mask.shape
    heights = [0] * W
    best = (0, 0, H, 0, W)
    for y in range(H):
        row = mask[y]
        heights = [h + 1 if row[x] else 0 for x, h in enumerate(heights)]
        stack = []
        for x in range(W + 1):
            h = heights[x] if x < W else 0
            start = x
            while stack and stack[-1][1] >= h:
                s, sh = stack.pop()
                area = sh * (x - s)
                if area > best[0]:
                    best = (area, y - sh + 1, y + 1, s, x)
                start = s
            stack.append((start, h))
    return best[1:]


def crop_black_borders(img, dark_thr=DARK_THR, min_area_frac=0.30):
    """Recorta al mayor rectángulo sin borde negro. Devuelve (img_recortada, (y0,y1,x0,x1), mask)."""
    H, W = img.shape[:2]
    mask = valid_mask(img, dark_thr)
    scale = min(1.0, 256 / max(H, W))                       # calculamos en pequeño por velocidad
    small = cv2.resize(mask.astype(np.uint8), None, fx=scale, fy=scale,
                       interpolation=cv2.INTER_NEAREST).astype(bool)
    y0, y1, x0, x1 = largest_rectangle(small)
    y0, y1 = int(np.ceil(y0 / scale)), int(np.floor(y1 / scale))
    x0, x1 = int(np.ceil(x0 / scale)), int(np.floor(x1 / scale))
    if (y1 - y0) * (x1 - x0) < min_area_frac * H * W:       # fallback: algo raro, no recortamos
        print("  [aviso] rectángulo válido muy pequeño, no se recorta")
        y0, y1, x0, x1 = 0, H, 0, W
    return img[y0:y1, x0:x1], (y0, y1, x0, x1), mask

### Paso 2 · Cuadrado central + resize

In [ ]:
def square_resize(img, size=TARGET_SIZE):
    H, W = img.shape[:2]
    s = min(H, W)
    y0, x0 = (H - s) // 2, (W - s) // 2
    sq = img[y0:y0 + s, x0:x0 + s]
    interp = cv2.INTER_AREA if s > size else cv2.INTER_CUBIC
    return cv2.resize(sq, (size, size), interpolation=interp)

### Paso 3 · Color constancy (Shades of Gray)

In [ ]:
def shades_of_gray(img, p=SOG_P):
    x = img.astype(np.float32)
    illum = np.power(np.mean(np.power(x, p), axis=(0, 1)), 1.0 / p)   # iluminante estimado (R,G,B)
    illum = illum / np.linalg.norm(illum)
    out = x / (illum * np.sqrt(3))
    return np.clip(out, 0, 255).astype(np.uint8)

### Paso 4 · Eliminación de pelos (DullRazor)
Blackhat con kernels lineales en varias orientaciones → umbral → **filtro de componentes** (solo largas y finas, para no borrar la red de pigmento de la lesión) → dilatación → inpainting.

In [ ]:
def line_kernel(length, angle_deg):
    k = np.zeros((length, length), np.uint8)
    c = length // 2
    a = np.deg2rad(angle_deg)
    dx, dy = np.cos(a) * c, np.sin(a) * c
    cv2.line(k, (int(round(c - dx)), int(round(c + dy))), (int(round(c + dx)), int(round(c - dy))), 1, 1)
    return k


def hair_mask(img, length=HAIR_LEN, thr=HAIR_THR, angles=range(0, 180, 30),
              min_len=25, min_aspect=3.5, dilate_iter=2):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    bh = np.maximum.reduce([cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, line_kernel(length, a)) for a in angles])
    raw = (bh > thr).astype(np.uint8)

    n, lab, stats, _ = cv2.connectedComponentsWithStats(raw, connectivity=8)
    keep = np.zeros_like(raw)
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if area < 15:
            continue
        sub = lab[y:y + h, x:x + w] == i
        pts = np.column_stack(np.where(sub))[:, ::-1].astype(np.float32)
        (_, _), (rw, rh), _ = cv2.minAreaRect(pts)
        length_c, width_c = max(rw, rh), max(min(rw, rh), 1.0)
        elongated = length_c >= min_len and length_c / width_c >= min_aspect
        thin_curve = max(w, h) >= 2 * min_len and area / (w * h) < 0.2   # pelos curvados
        if elongated or thin_curve:
            keep[y:y + h, x:x + w][sub] = 1
    return cv2.dilate(keep, np.ones((3, 3), np.uint8), iterations=dilate_iter)


def remove_hair(img):
    m = hair_mask(img)
    clean = cv2.inpaint(img, m * 255, 3, cv2.INPAINT_TELEA)
    return clean, m

### Paso 5 · Dos copias
- **`img_feat`**: sin blur → de aquí se sacan las features de color y textura.
- **`img_seg`**: con blur → **solo** para segmentar (la máscara resultante se aplica luego sobre `img_feat`).

In [ ]:
def make_blurred(img, k=BLUR_KSIZE):
    return cv2.medianBlur(img, k)      # preserva bordes mejor que un gaussiano

### Pipeline completo (guarda cada paso)

In [ ]:
def preprocess(img):
    steps = {"original": img}
    cropped, rect, bmask = crop_black_borders(img)
    steps["rect"], steps["border_mask"] = rect, bmask
    steps["cropped"] = cropped
    steps["resized"] = square_resize(cropped)
    steps["color_const"] = shades_of_gray(steps["resized"])
    steps["img_feat"], steps["hair_mask"] = remove_hair(steps["color_const"])   # sin blur
    steps["img_seg"] = make_blurred(steps["img_feat"])                          # con blur
    return steps

## Visualización paso a paso

In [ ]:
def show_steps(label, name, s):
    H, W = s["original"].shape[:2]
    y0, y1, x0, x1 = s["rect"]
    hair_pct = 100 * s["hair_mask"].mean()

    fig, axs = plt.subplots(2, 4, figsize=(20, 10))
    axs = axs.ravel()

    # 0. original + rectángulo detectado
    ov = s["original"].copy()
    ov[~s["border_mask"]] = (0.5 * ov[~s["border_mask"]] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
    cv2.rectangle(ov, (x0, y0), (x1 - 1, y1 - 1), (0, 255, 0), max(2, W // 200))
    axs[0].imshow(ov);  axs[0].set_title(f"Original {W}x{H}\n(rojo = borde detectado, verde = recorte)")

    axs[1].imshow(s["cropped"]);      axs[1].set_title(f"1) Sin bordes negros {s['cropped'].shape[1]}x{s['cropped'].shape[0]}")
    axs[2].imshow(s["resized"]);      axs[2].set_title(f"2) Cuadrado + resize {TARGET_SIZE}x{TARGET_SIZE}")
    axs[3].imshow(s["color_const"]);  axs[3].set_title(f"3) Shades of Gray (p={SOG_P})")

    axs[4].imshow(s["hair_mask"], cmap="gray");  axs[4].set_title(f"4a) Máscara de pelos ({hair_pct:.2f}% de píxeles)")
    axs[5].imshow(s["img_feat"]);     axs[5].set_title("4b) Sin pelos  →  img_feat (SIN blur)")
    axs[6].imshow(s["img_seg"]);      axs[6].set_title(f"5) Median blur k={BLUR_KSIZE}  →  img_seg (CON blur)")

    c, r = TARGET_SIZE // 2, TARGET_SIZE // 8
    zoom = np.hstack([s["img_feat"][c - r:c + r, c - r:c + r], s["img_seg"][c - r:c + r, c - r:c + r]])
    axs[7].imshow(zoom, interpolation="nearest");  axs[7].set_title("Zoom centro: sin blur | con blur")

    for a in axs:
        a.axis("off")
    fig.suptitle(f"[{label}] {name}", fontsize=16, y=0.99)
    plt.tight_layout()
    plt.show()


results = []
for label, path in samples:
    print(f"Procesando {label}/{path.name} ...")
    s = preprocess(read_rgb(path))
    results.append((label, path.name, s))
    show_steps(label, path.name, s)

## Resumen: entrada vs. salida final

In [ ]:
n = len(results)
fig, axs = plt.subplots(3, n, figsize=(4.5 * n, 13))
for j, (label, name, s) in enumerate(results):
    for i, (key, t) in enumerate([("original", "original"), ("img_feat", "img_feat (sin blur)"), ("img_seg", "img_seg (con blur)")]):
        axs[i, j].imshow(s[key]); axs[i, j].axis("off")
        axs[i, j].set_title(f"{label}/{name}\n{t}", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
def border_ring(shape, frac=0.08):
    H, W = shape
    m = int(frac * min(H, W))
    ring = np.ones((H, W), bool)
    ring[m:H - m, m:W - m] = False
    return ring


def get_channels(img):
    """Canales candidatos (uint8) para separar lesión y piel."""
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    ycc = cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    # primera componente principal de los píxeles RGB
    X = img.reshape(-1, 3).astype(np.float32)
    X -= X.mean(0)
    _, _, Vt = np.linalg.svd(X[::10], full_matrices=False)
    pca = (X @ Vt[0]).reshape(img.shape[:2])
    pca = cv2.normalize(pca, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return {"gray": cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), "L": lab[..., 0], "a": lab[..., 1], "b": lab[..., 2],
            "Cr": ycc[..., 1], "Cb": ycc[..., 2], "S": hsv[..., 1], "V": hsv[..., 2],
            "R": img[..., 0], "G": img[..., 1], "B": img[..., 2], "pca": pca}


def otsu_mask(ch):
    """Otsu sobre un canal. El 'foreground' es la clase que menos aparece en el borde de la imagen (la piel)."""
    ch = cv2.normalize(ch, None, 0, 255, cv2.NORM_MINMAX)
    _, bw = cv2.threshold(ch, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bw = bw > 0
    if bw[border_ring(bw.shape)].mean() > 0.5:
        bw = ~bw
    return bw

In [ ]:
def clean_mask(bw, open_k=5, close_k=15, center_sigma=0.35):
    """Apertura + cierre, elige el componente grande y cercano al centro, y rellena huecos."""
    ell = lambda n: cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (n, n))
    m = bw.astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, ell(open_k))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, ell(close_k))
    n, lab, stats, cent = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n == 1:
        return np.zeros_like(bw)
    S = m.shape[0]
    c = np.array([m.shape[1] / 2, m.shape[0] / 2])
    score = [stats[i, cv2.CC_STAT_AREA] * np.exp(-(np.linalg.norm(cent[i] - c) / (center_sigma * S)) ** 2)
             for i in range(1, n)]
    best = int(np.argmax(score)) + 1
    m = (lab == best).astype(np.uint8)
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)   # rellena huecos internos
    filled = np.zeros_like(m)
    cv2.drawContours(filled, cnts, -1, 1, thickness=-1)
    return filled.astype(bool)


def segment(img_seg, channel="Cr"):
    return clean_mask(otsu_mask(get_channels(img_seg)[channel]))


def contour_overlay(img, mask, color=(0, 255, 0), thick=2):
    out = img.copy()
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(out, cnts, -1, color, thick)
    return out

In [ ]:
CANDIDATES = ["L", "a", "b", "Cr", "S", "pca"]

fig, axs = plt.subplots(len(results), len(CANDIDATES), figsize=(3.2 * len(CANDIDATES), 3.4 * len(results)))
for i, (label, name, s) in enumerate(results):
    for j, ch in enumerate(CANDIDATES):
        m = segment(s["img_seg"], ch)
        axs[i, j].imshow(contour_overlay(s["img_feat"], m))
        axs[i, j].set_title(f"{name}\n{ch}  ({100 * m.mean():.0f}% área)", fontsize=9)
        axs[i, j].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
CHANNEL = "S"     # el que mejor os parezca en el chunk anterior


def shape_features(mask):
    m = mask.astype(np.uint8)
    cnt = max(cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0], key=cv2.contourArea)
    area, perim = cv2.contourArea(cnt), cv2.arcLength(cnt, True)
    hull_area = cv2.contourArea(cv2.convexHull(cnt))
    (_, _), (w, h), _ = cv2.fitEllipse(cnt)
    return {"area_frac": float(mask.mean()),
            "circularity": 4 * np.pi * area / perim ** 2,
            "solidity": area / hull_area,
            "eccentricity": float(np.sqrt(1 - (min(w, h) / max(w, h)) ** 2)),
            "touches_border": bool(mask[0].any() or mask[-1].any() or mask[:, 0].any() or mask[:, -1].any())}


fig, axs = plt.subplots(len(results), 3, figsize=(12, 4 * len(results)))
for i, (label, name, s) in enumerate(results):
    mask = segment(s["img_seg"], CHANNEL)
    s["mask"] = mask
    axs[i, 0].imshow(contour_overlay(s["img_feat"], mask)); axs[i, 0].set_title(f"[{label}] {name}")
    axs[i, 1].imshow(mask, cmap="gray");                   axs[i, 1].set_title("máscara")
    axs[i, 2].imshow(s["img_feat"] * mask[..., None]);     axs[i, 2].set_title("lesión aislada")
    for a in axs[i]: a.axis("off")
    print(name, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in shape_features(mask).items()})
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.cluster import KMeans
from skimage.segmentation import morphological_chan_vese

ell = lambda n: cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (n, n))


def seg_otsu(img_seg, channel="S"):
    return segment(img_seg, channel)


def seg_kmeans(img_seg, k=2):
    """K-means sobre los colores Lab. Lesión = cluster que menos aparece en el borde (piel)."""
    lab = cv2.cvtColor(img_seg, cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(np.float32)
    km = KMeans(n_clusters=k, n_init=3, random_state=0).fit(lab[::7])
    lbl = km.predict(lab).reshape(img_seg.shape[:2])
    ring = border_ring(lbl.shape)
    skin = np.argmax([(lbl[ring] == c).mean() for c in range(k)])
    return clean_mask(lbl != skin)


def seg_grabcut(img_seg, init_mask=None, band=15, iters=5):
    """GrabCut inicializado con una máscara previa (Otsu): refina la frontera con modelos de color."""
    if init_mask is None:
        init_mask = seg_otsu(img_seg)
    m = init_mask.astype(np.uint8)
    gc = np.full(m.shape, cv2.GC_BGD, np.uint8)
    gc[cv2.dilate(m, ell(2 * band + 1)) > 0] = cv2.GC_PR_BGD      # banda exterior: dudoso
    gc[m > 0] = cv2.GC_PR_FGD                                      # dentro: probable lesión
    gc[cv2.erode(m, ell(2 * band + 1)) > 0] = cv2.GC_FGD           # núcleo: seguro lesión
    bgd, fgd = np.zeros((1, 65)), np.zeros((1, 65))
    cv2.grabCut(img_seg, gc, None, bgd, fgd, iters, cv2.GC_INIT_WITH_MASK)
    return clean_mask((gc == cv2.GC_FGD) | (gc == cv2.GC_PR_FGD))


def seg_watershed(img_seg, init_mask=None, band=15):
    """Watershed con marcadores: núcleo de lesión, fondo seguro, y la zona intermedia se decide por gradiente."""
    if init_mask is None:
        init_mask = seg_otsu(img_seg)
    m = init_mask.astype(np.uint8)
    markers = np.zeros(m.shape, np.int32)
    markers[cv2.dilate(m, ell(2 * band + 1)) == 0] = 1             # fondo seguro
    markers[cv2.erode(m, ell(2 * band + 1)) > 0] = 2               # lesión segura
    cv2.watershed(img_seg.copy(), markers)
    return clean_mask(markers == 2)


def seg_chanvese(img_seg, channel="S", iters=80, smoothing=3):
    """Contorno activo Chan-Vese (morfológico), inicializado con Otsu."""
    ch = cv2.normalize(get_channels(img_seg)[channel], None, 0, 1, cv2.NORM_MINMAX, cv2.CV_32F)
    ls = seg_otsu(img_seg, channel)
    out = morphological_chan_vese(ch, num_iter=iters, init_level_set=ls, smoothing=smoothing).astype(bool)
    if out[border_ring(out.shape)].mean() > 0.5:
        out = ~out
    return clean_mask(out)


METHODS = {"Otsu(S)": seg_otsu, "K-means": seg_kmeans, "GrabCut": seg_grabcut,
           "Watershed": seg_watershed, "Chan-Vese": seg_chanvese}

jacc = lambda a, b: (a & b).sum() / max((a | b).sum(), 1)

fig, axs = plt.subplots(len(results), len(METHODS), figsize=(3.4 * len(METHODS), 3.5 * len(results)))
for i, (label, name, s) in enumerate(results):
    masks = {k: f(s["img_seg"]) for k, f in METHODS.items()}
    s["masks"] = masks
    for j, (k, m) in enumerate(masks.items()):
        axs[i, j].imshow(contour_overlay(s["img_feat"], m))
        axs[i, j].set_title(f"{name}\n{k}  ({100 * m.mean():.0f}% | J vs Otsu {jacc(m, masks['Otsu(S)']):.2f})", fontsize=9)
        axs[i, j].axis("off")
plt.tight_layout(); plt.show()